In [17]:
import tifffile
import matplotlib.pyplot as plt
import napari
from pathlib import Path
from liffile import LifFile
import numpy as np
from skimage.filters import gaussian

In [31]:
input_dir = Path(r"Z:\Bel")


def lif_pixel_sizes(img):
    """Return (z_um, y_um, x_um) from a liffile image; any value is None if unavailable."""
    z_um = y_um = x_um = None
    try:
        coords = img.asxarray().coords
        if "X" in coords and coords["X"].size >= 2:
            x_um = abs(float(coords["X"][1] - coords["X"][0])) * 1e6
        if "Y" in coords and coords["Y"].size >= 2:
            y_um = abs(float(coords["Y"][1] - coords["Y"][0])) * 1e6
        if "Z" in coords and coords["Z"].size >= 2:
            z_um = abs(float(coords["Z"][1] - coords["Z"][0])) * 1e6
    except Exception as e:
        print(f"  [WARN] could not read LIF pixel sizes: {e}")
    return z_um, y_um, x_um


def to_tczyx(arr, dims):
    """Convert lif image array to (T, C, Z, Y, X) for consistent indexing."""
    data = arr
    dim_list = list(dims)

    for needed in ("T", "C", "Z", "Y", "X"):
        if needed not in dim_list:
            data = data[np.newaxis, ...]
            dim_list = [needed] + dim_list

    order = [dim_list.index(a) for a in ("T", "C", "Z", "Y", "X")]
    return np.transpose(data, order)


for lif_path in sorted(input_dir.glob("*.lif")):
    print(f"LIF: {lif_path}")
    with LifFile(lif_path) as lif:
        z_um, y_um, x_um = lif_pixel_sizes(lif)
        if len(lif.images) == 4:
            print("  SKIP - no images in file")
            continue

        img = lif.images[0]  # first image in the LIF
        print(f"  image path: {''.join(img.path)}")
        print(f"  dims: {tuple(img.dims)}, shape: {img.shape}")

        data = to_tczyx(img.asarray(), tuple(img.dims))  # (T, C, Z, Y, X)
        n_t, n_c, n_z, _, _ = data.shape
        print(f"  normalized shape (T,C,Z,Y,X): {data.shape}")

        if n_t < 2:
            print("  SKIP - need at least 2 timepoints (t0 and t1)")
            continue
        if n_c < 2:
            print("  SKIP - need at least 2 channels (c0 dextran, c1 brightfield)")
            continue

        # Channel convention from your note: c0=dextran, c1=brightfield
        dex_t0 = data[0, 0]  # (Z, Y, X)
        dex_t1 = data[1, 0]  # (Z, Y, X)
        dex_t2 = data[2,0]
        bf_t1 = data[1, 1]   # (Z, Y, X)
        
        z_um, y_um, x_um = lif_pixel_sizes(img)

        # Napari expects per-axis scale matching data dimensions (here Z, Y, X).
        scale_zyx = (
            float(z_um) if z_um is not None else 1.0,
            float(y_um) if y_um is not None else 1.0,
            float(x_um) if x_um is not None else 1.0,
        )

        # Signed change: positive=increased dextran, negative=decreased dextran
        smooth_delta_t1_t0 = gaussian(dex_t1.astype(np.float32) - dex_t0.astype(np.float32), sigma = 3)
        smooth_delta_t2_t1 = gaussian(dex_t2.astype(np.float32) - dex_t1.astype(np.float32), sigma = 3)
        min_value = np.min([np.min(smooth_delta_t1_t0), np.min(smooth_delta_t2_t1)])
        max_value = np.min([np.max(smooth_delta_t1_t0), np.max(smooth_delta_t2_t1)])
        print(min_value, max_value)

        viewer = napari.Viewer()
        viewer.add_image(bf_t1, name="brightfield t1", colormap="gray", scale=scale_zyx)
        viewer.add_image(smooth_delta_t1_t0, name="change t1-t0 (dextran)", colormap="bwr", contrast_limits=(min_value, max_value), opacity=0.55, scale=scale_zyx)
        viewer.add_image(smooth_delta_t2_t1, name="change t2-t1 (dextran)", colormap="bwr", contrast_limits=(min_value, max_value), opacity=0.55, scale=scale_zyx)


        break


LIF: Z:\Bel\Permeability_for_heatmap_DZ.lif
  [WARN] could not read LIF pixel sizes: 'LifFile' object has no attribute 'asxarray'
  image path: WW
  dims: ('T', 'C', 'Z', 'Y', 'X'), shape: (3, 2, 22, 512, 512)
  normalized shape (T,C,Z,Y,X): (3, 2, 22, 512, 512)
-2.7603126 1.3617544


In [ ]:
# Channel convention: c0 = dextran, c1 = brightfield